In [1]:
# ── Cell 1: installs ──────────────────────────────────────────────────
!pip install transformers torch scikit-learn pandas openpyxl --quiet

In [2]:
# ── Cell 2: imports + device ─────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# If this prints "cpu", go to Runtime > Change runtime type > GPU (T4 is fine)
# before continuing — CPU will work but embedding 728 sequences will be slower.

Using device: cuda


In [3]:
# ── Cell 3: load your dataset ────────────────────────────────────────
# Upload dataset.xlsx via the Colab file browser (left sidebar, folder icon),
# or mount Drive if that's where it lives:
# from google.colab import drive
# drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()   # pick dataset.xlsx when the file picker appears

file_path = list(uploaded.keys())[0]   # or hardcode the exact filename/path
xls = pd.ExcelFile(file_path)
dfs = [pd.read_excel(xls, sheet_name=s) for s in xls.sheet_names]
data = pd.concat(dfs, ignore_index=True)
data["length"] = data["Sequence"].str.len()
print("Dataset shape:", data.shape)
print(data.head())

Saving dataset.xlsx to dataset (1).xlsx
Dataset shape: (728, 3)
  Sequence  log(IC50)  length
0       AA   1.710963       2
1       AF   2.278754       2
2       AG   3.397940       2
3       AH   2.718917       2
4       AI   0.532754       2


In [4]:
# ── Cell 4: load frozen ESM-2 ─────────────────────────────────────────
ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

esm_tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)
esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME).to(device).eval()
for p in esm_model.parameters():
    p.requires_grad = False

print("ESM-2 loaded. Hidden size:", esm_model.config.hidden_size)  # 320 for the 8M model

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM-2 loaded. Hidden size: 320


In [5]:
# ── Cell 5: embedding extraction functions ─────────────────────────────
def prep_sequence_esm(seq: str) -> str:
    """ESM tokenizers handle amino-acid-level tokenization natively —
    no space-separation needed (unlike ProtBERT). Just clean up rare/
    ambiguous residues for consistency."""
    return seq.upper().replace('U', 'X').replace('Z', 'X').replace('O', 'X').replace('B', 'X')


@torch.no_grad()
def embed_sequences_esm(seqs, batch_size=16, pooling="mean"):
    all_embeddings = []
    prepped = [prep_sequence_esm(s) for s in seqs]

    for i in range(0, len(prepped), batch_size):
        batch = prepped[i:i + batch_size]
        enc = esm_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64)
        enc = {k: v.to(device) for k, v in enc.items()}
        out = esm_model(**enc)
        last_hidden = out.last_hidden_state

        if pooling == "cls":
            pooled = last_hidden[:, 0, :]
        else:
            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            pooled = summed / counts

        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)

In [6]:
# ── Cell 6: run embedding extraction on your full dataset ─────────────
esm_embeddings = embed_sequences_esm(data["Sequence"].tolist(), batch_size=16, pooling="mean")
print("ESM-2 embedding matrix shape:", esm_embeddings.shape)   # (728, 320)

np.save('esm_embeddings.npy', esm_embeddings)   # cache so you don't recompute if the runtime restarts

ESM-2 embedding matrix shape: (728, 320)


In [7]:
# ── Cell 7: PCA + Ridge pipeline factory ────────────────────────────────
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

def make_pca_pipeline(n_components=30):
    return make_pipeline(
        StandardScaler(),
        PCA(n_components=n_components, random_state=42),
        Ridge(alpha=1.0),
    )

In [8]:
# ── Cell 8: length-stratified repeated CV ──────────────────────────────
from sklearn.model_selection import RepeatedKFold, cross_validate

cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=42)
esm_cols = [f'esm_{i}' for i in range(esm_embeddings.shape[1])]
esm_df = pd.DataFrame(esm_embeddings, columns=esm_cols, index=data.index)
data_esm = pd.concat([data, esm_df], axis=1)

print('=== ESM-2 embedding Ridge (with PCA), length-stratified ===')
for L, grp in data_esm.groupby('length'):
    if len(grp) < 30:
        continue
    X = grp[esm_cols]
    y = grp['log(IC50)']
    n_comp = min(30, len(grp) - 20, esm_embeddings.shape[1] - 1)
    reg_model = make_pca_pipeline(n_components=n_comp)
    scores = cross_validate(reg_model, X, y, cv=cv, scoring='r2')
    print(f'length={L:2d} n={len(grp):4d}  R2={scores["test_score"].mean():+.4f} '
          f'+/- {scores["test_score"].std():.4f}')

=== ESM-2 embedding Ridge (with PCA), length-stratified ===
length= 2 n= 133  R2=+0.2867 +/- 0.2169
length= 3 n= 215  R2=-0.0893 +/- 0.1830
length= 4 n=  90  R2=-0.2998 +/- 0.5597
length= 5 n= 132  R2=-0.1511 +/- 0.2940
length= 6 n= 114  R2=-0.3606 +/- 0.3202


In [9]:
# ── Cell 9: cosine-similarity diagnostics ──────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

for L in [2, 3, 6]:
    mask = data['length'] == L
    embs = esm_embeddings[mask.values]
    sim = cosine_similarity(embs)
    off_diag = sim[np.triu_indices_from(sim, k=1)]
    print(f'ESM-2 length={L}: mean cosine sim={off_diag.mean():.4f}, '
          f'std={off_diag.std():.4f}, min={off_diag.min():.4f}, max={off_diag.max():.4f}')

seqs_check = ['LL', 'DD', 'RR', 'GG']
check_emb = embed_sequences_esm(seqs_check, pooling='mean')
print(pd.DataFrame(cosine_similarity(check_emb), index=seqs_check, columns=seqs_check).round(3))

ESM-2 length=2: mean cosine sim=0.9379, std=0.0270, min=0.8179, max=0.9977
ESM-2 length=3: mean cosine sim=0.9466, std=0.0254, min=0.7990, max=0.9992
ESM-2 length=6: mean cosine sim=0.9482, std=0.0275, min=0.7915, max=0.9988
       LL     DD     RR     GG
LL  1.000  0.913  0.919  0.904
DD  0.913  1.000  0.877  0.914
RR  0.919  0.877  1.000  0.929
GG  0.904  0.914  0.929  1.000
